# 0 · Extract Data — DDI-2013 → Holistic NLI Train/Val + Raw Test Manifest

**Fixes applied in this version (see changelog at the bottom of this notebook for full detail):**
1. Every generated row now gets a stable `original_id` (needed by the atomic pipeline and by
   `AtomicAggregator` downstream in `MAIN_NB`, which groups on `original_id`).
2. A `scenario` column is now emitted alongside `label`. `MAIN_NB.NLIDataBundle.summary()`
   reads `df['scenario']` for the class-distribution printout — the previous version of this
   notebook only wrote `label`, so that printout was silently reading a column that didn't
   exist in this notebook's own output (it only worked because some other, undocumented step
   added it later). For the holistic set, `scenario == label` by construction — there is no
   4th "fake_drug" scenario here (see note in §4 below on why that's intentional).
3. An `entity_valid` column is added (always `True` for this template-based holistic set,
   since every hypothesis is built from a real DrugBank/MedLine entity pair). This exists
   purely for **schema parity** with `holistic_test.csv` (built in `1_AugmentResponses.ipynb`),
   where `entity_valid` is *not* trivially true — that's where the fake-drug/hallucination
   signal actually lives. Folding a 4th learned class into the classifier for something that's
   a deterministic lookup problem was the root cause of the fake_drug-class distortion identified
   in the MAIN_NB results review; see §4.
4. `parse_ddi_xml_dir` now raises a clear `FileNotFoundError` instead of silently returning an
   empty DataFrame when a corpus directory is missing — a wrong path used to fail silently
   several cells later with a confusing `KeyError`/`ValueError` instead of a diagnosable error
   at the source.
5. Output filenames renamed to `holistic_train.csv` / `holistic_val.csv` to match the
   `{prefix}_train.csv` / `{prefix}_val.csv` convention `MAIN_NB.NLIDataBundle` actually expects
   (`prefix="holistic"`). The previous filenames (`train_augmented_nli.csv`, `val_augmented_nli.csv`)
   didn't match what `MAIN_NB` was reading and had to be renamed manually out-of-band.


In [1]:
import os
import itertools
import xml.etree.ElementTree as ET

import pandas as pd
from sklearn.model_selection import train_test_split


## 1 · Synthetic hypothesis templates

Unchanged from the original — one bank of entailment/contradiction/neutral hypothesis templates per DDI interaction type (`mechanism`, `effect`, `advise`, `int`).

In [2]:
augmented_templates = {
    'mechanism': {
        'entailment': [
            "The concurrent use of {e1} and {e2} leads to significant alterations in systemic clearance rates.",
            "Co-administration of {e1} and {e2} produces a marked shift in their expected pharmacokinetic profiles.",
            "Combined therapy with {e1} and {e2} results in pronounced fluctuations in plasma concentrations.",
            "The metabolic pathways of both {e1} and {e2} undergo mutual interference upon simultaneous ingestion.",
            "An atypical area under the curve (AUC) is routinely observed during the concurrent administration of {e1} and {e2}.",
            "The bioavailability parameters of both compounds are modified when {e1} and {e2} are utilized together.",
            "Concomitant exposure to {e1} and {e2} triggers a measurable disruption in normal hepatic enzyme activity.",
            "Renal excretion kinetics exhibit notable deviations when {e1} and {e2} are present in the system simultaneously.",
            "The free fraction in systemic circulation is altered due to competitive protein binding between {e1} and {e2}.",
            "Half-life extensions are a recognized consequence of the pharmacokinetic interaction between {e1} and {e2}.",
            "Co-administration of {e1} and {e2} results in altered metabolic clearance rates and changes in plasma concentrations.",
            "Concurrent use of {e1} and {e2} triggers a significant alteration in CYP-mediated metabolic pathways and AUC.",
            "The exposure profile of both compounds is modified when {e1} and {e2} are combined due to impaired hepatic metabolism.",
            "The interaction between {e1} and {e2} is driven by altered enzyme activity affecting drug clearance.",
            "Concomitant exposure to {e1} and {e2} impacts renal clearance and modifies elimination half-life parameters.",
            "Concurrent use of {e1} and {e2} produces higher systemic concentrations through pharmacokinetic interference.",
            "Biotransformation pathways undergo alteration when {e1} and {e2} are present concurrently, affecting plasma levels.",
            "The pharmacokinetic profile is significantly modified when {e1} and {e2} are administered alongside each other.",
            "Enzymatic induction or inhibition occurs during concurrent use of {e1} and {e2}, shifting serum levels.",
            "A clinically relevant shift in plasma exposure parameters occurs when {e1} and {e2} are taken concurrently."
        ],
        'contradiction': [
            "Current pharmacokinetic data does not indicate a significant metabolic interaction between {e1} and {e2}.",
            "There is insufficient evidence to suggest that co-administration of {e1} and {e2} alters respective clearance rates.",
            "Clinical studies have generally failed to demonstrate anomalous plasma concentrations when combining {e1} and {e2}.",
            "The bioavailability of {e1} and {e2} appears to remain stable during concomitant therapy.",
            "Routine monitoring rarely reveals meaningful changes in the AUC for either {e1} or {e2} when used concurrently.",
            "Hepatic enzyme assays do not currently support a strong metabolic interference between {e1} and {e2}.",
            "The existing literature lacks definitive proof that combining {e1} and {e2} impacts renal excretion pathways.",
            "Alterations in protein binding affinity are not typically observed following the joint administration of {e1} and {e2}.",
            "It is generally understood that the elimination half-lives of {e1} and {e2} are unaffected by their combined use.",
            "There is limited pharmacological basis to expect significant pharmacokinetic shifts when {e1} and {e2} are co-administered.",
            "Current pharmacokinetic data does not indicate a significant metabolic interaction between {e1} and {e2}.",
            "There is insufficient evidence to suggest that co-administration of {e1} and {e2} alters respective clearance rates.",
            "Clinical studies have generally failed to demonstrate anomalous plasma concentrations when combining {e1} and {e2}.",
            "The bioavailability of {e1} and {e2} appears to remain stable during concomitant therapy.",
            "Routine monitoring rarely reveals meaningful changes in the AUC for either {e1} or {e2} when used concurrently.",
            "Hepatic enzyme assays do not currently support a strong metabolic interference between {e1} and {e2}.",
            "The existing literature lacks definitive proof that combining {e1} and {e2} impacts renal excretion pathways.",
            "Alterations in protein binding affinity are not typically observed following the joint administration of {e1} and {e2}.",
            "It is generally understood that the elimination half-lives of {e1} and {e2} are unaffected by their combined use.",
            "There is limited pharmacological basis to expect significant pharmacokinetic shifts when {e1} and {e2} are co-administered."
        ],
        'neutral': [
            "Both {e1} and {e2} are widely dispensed in outpatient pharmacy settings across various regions.",
            "The synthesis processes for {e1} and {e2} involve advanced chemical extraction methodologies.",
            "Commercial formulations of {e1} and {e2} frequently share similar ambient storage requirements.",
            "Pharmaceutical distribution networks handle the logistics for both {e1} and {e2} on a global scale.",
            "The basic molecular weight classifications of {e1} and {e2} fall within standard therapeutic ranges.",
            "Primary literature discussing {e1} and {e2} can be readily accessed through major medical databases.",
            "Oral solid dosage forms are the predominant method of delivery for both {e1} and {e2}.",
            "{e1} and {e2} were developed following extensive high-throughput compound screening protocols.",
            "The regulatory approval timelines for {e1} and {e2} span multiple decades of clinical research.",
            "Synthesizing {e1} and {e2} requires stringent adherence to international manufacturing quality standards.",
            "{e1} and {e2} are both available in oral dosage formulations.",
            "{e1} was approved for clinical use before {e2}.",
            "{e2} is frequently prescribed in outpatient settings alongside unrelated therapies.",
            "Both {e1} and {e2} are listed in several international drug formularies.",
            "{e1} and {e2} may be manufactured by different pharmaceutical companies.",
            "{e2} is commonly dispensed in tablet form, whereas {e1} may also be injectable.",
            "The generic versions of {e1} and {e2} are widely distributed globally.",
            "{e1} and {e2} have been referenced in pharmacoeconomic studies.",
            "Patients receiving {e1} are often older than those typically prescribed {e2}.",
            "The labeling information for {e1} and {e2} is periodically updated by regulators."]
    },
    'effect': {
        'entailment': [
            "The concurrent administration of {e1} and {e2} is associated with a heightened risk of adverse clinical outcomes.",
            "Combined therapy involving {e1} and {e2} frequently precipitates severe physiological toxicity.",
            "A synergistic magnification of untoward physiological responses is noted when {e1} and {e2} are co-administered.",
            "Patients receiving both {e1} and {e2} exhibit an increased incidence of generalized adverse events.",
            "The pharmacodynamic interplay between {e1} and {e2} exacerbates baseline clinical risks.",
            "Concomitant use of {e1} and {e2} leads to a documented deterioration in overall patient tolerability.",
            "The combination of {e1} and {e2} routinely triggers compounded systemic side effects.",
            "Utilizing {e1} and {e2} together worsens the expected adverse event profile compared to monotherapy.",
            "Severe multi-systemic physiological reactions are more prevalent during the simultaneous use of {e1} and {e2}.",
            "Co-prescribing {e1} and {e2} significantly elevates the probability of complex clinical complications.",
            "Using {e1} together with {e2} increases the risk of serious adverse reactions.",
            "The combination of {e1} and {e2} may precipitate clinically significant toxicity.",
            "Patients receiving both {e1} and {e2} are at elevated risk for symptomatic side effects.",
            "Co-administration of {e1} with {e2} has been associated with severe physiological responses.",
            "The interaction between {e1} and {e2} can lead to heightened adverse clinical outcomes.",
            "Concomitant use of {e1} and {e2} leads to a documented deterioration in overall patient tolerability.",
            "The combination of {e1} and {e2} routinely triggers compounded systemic side effects.",
            "The risk of severe multi-systemic physiological complications rises substantially when {e1} is combined with {e2}.",
            "Patients treated with both {e1} and {e2} may experience amplified toxic effects.",
            "Administration of {e1} together with {e2} may worsen patient safety outcomes."
        ],
        'contradiction': [
            "Current data does not suggest an elevated risk of adverse clinical outcomes when combining {e1} and {e2}.",
            "There is limited evidence indicating that the concurrent use of {e1} and {e2} worsens physiological toxicity.",
            "Clinical observations generally do not report synergistic adverse events involving {e1} and {e2}.",
            "The overall side effect profile appears stable during the concomitant administration of {e1} and {e2}.",
            "Existing literature lacks consistent proof that {e1} and {e2} exacerbate generalized systemic risks.",
            "It is unlikely that the simultaneous use of {e1} and {e2} leads to compounded clinical complications.",
            "Patient tolerability is typically maintained when {e1} and {e2} are utilized in combination therapy.",
            "There is insufficient documentation to confirm a severe pharmacodynamic clash between {e1} and {e2}.",
            "Pharmacovigilance databases rarely flag worsened overall symptom severity for the {e1} and {e2} pairing.",
            "The hypothesized increase in systemic adverse reactions between {e1} and {e2} remains largely unsubstantiated.",
            "Current data does not suggest an elevated risk of adverse clinical outcomes when combining {e1} and {e2}.",
            "There is limited evidence indicating that the concurrent use of {e1} and {e2} worsens physiological toxicity.",
            "Clinical observations generally do not report synergistic adverse events involving {e1} and {e2}.",
            "The overall side effect profile appears stable during the concomitant administration of {e1} and {e2}.",
            "Existing literature lacks consistent proof that {e1} and {e2} exacerbate generalized systemic risks.",
            "It is unlikely that the simultaneous use of {e1} and {e2} leads to compounded clinical complications.",
            "Patient tolerability is typically maintained when {e1} and {e2} are utilized in combination therapy.",
            "There is insufficient documentation to confirm a severe pharmacodynamic clash between {e1} and {e2}.",
            "Pharmacovigilance databases rarely flag worsened overall symptom severity for the {e1} and {e2} pairing.",
            "The hypothesized increase in systemic adverse reactions between {e1} and {e2} remains largely unsubstantiated."
        ],
        'neutral': [
            "Hospital billing systems utilize distinct alphanumeric coding for the administration of {e1} and {e2}.",
            "The direct-to-consumer marketing strategies for {e1} and {e2} emphasize patient quality of life.",
            "The proprietary naming conventions for {e1} and {e2} follow standard industry linguistics.",
            "Generic equivalents for both {e1} and {e2} are subject to routine patent expiration cycles.",
            "National formularies categorize {e1} and {e2} based on broad therapeutic utility.",
            "Institutional pharmacy shelving guidelines often place {e1} and {e2} in separate inventory sections.",
            "Packaging materials for commercial batches of {e1} and {e2} are sourced from certified suppliers.",
            "The historical development timelines for {e1} and {e2} are frequently highlighted in academic reviews.",
            "Tiered insurance coverage models typically apply varying co-pay structures to {e1} and {e2}.",
            "Transporting bulk quantities of {e1} and {e2} relies on established commercial freight networks.",
            "{e1} and {e2} are both mentioned in current clinical treatment guidelines.",
            "{e2} is commonly prescribed in primary care settings, whereas {e1} is often initiated in hospitals.",
            "The packaging size of {e1} differs from that typically used for {e2}.",
            "{e1} and {e2} are available under multiple brand names worldwide.",
            "Research publications involving {e1} and {e2} have increased over the last decade.",
            "{e1} is frequently included in electronic prescribing databases alongside {e2}.",
            "{e2} has been studied extensively in elderly populations compared with {e1}.",
            "Both {e1} and {e2} may require dose adjustments in specific patient groups.",
            "{e1} and {e2} are stocked in many tertiary-care hospital pharmacies.",
            "The regulatory documentation for {e1} and {e2} differs across countries."
        ]
    },
    'advise': {
        'entailment': [
            "Clinical guidelines dictate that the concurrent use of {e1} and {e2} requires intensive clinical monitoring.",
            "Co-administration of {e1} and {e2} is strongly contraindicated in modern therapeutic protocols.",
            "Healthcare providers are advised to exercise extreme caution before combining {e1} and {e2}.",
            "The simultaneous prescription of {e1} and {e2} should be generally avoided due to overlapping risk profiles.",
            "Authoritative medical directives recommend pursuing alternative therapies over the {e1} and {e2} combination.",
            "Concomitant therapy with {e1} and {e2} necessitates a proactive reduction in baseline dosages.",
            "It is considered poor clinical practice to initiate a combination regimen consisting of {e1} and {e2}.",
            "Practitioners must continuously evaluate risk versus benefit when maintaining patients on both {e1} and {e2}.",
            "High-frequency diagnostic surveillance is mandated when {e1} and {e2} are utilized concurrently.",
            "Current prescribing information carries explicit warnings against the simultaneous use of {e1} and {e2}.",
            "Concurrent use of {e1} and {e2} should generally be avoided.",
            "Clinicians are advised not to prescribe {e1} together with {e2}.",
            "The combination of {e1} and {e2} is contraindicated due to interaction concerns.",
            "Caution is strongly recommended when administering {e1} alongside {e2}.",
            "Guidelines advise against co-administration of {e1} with {e2}.",
            "Patients receiving {e1} should not ordinarily be started on {e2}.",
            "The use of {e1} in combination with {e2} is not recommended in routine practice.",
            "Healthcare professionals should carefully monitor or avoid the combination of {e1} and {e2}.",
            "Prescribers are instructed to use extreme caution if {e1} and {e2} must be combined.",
            "Co-prescription of {e1} with {e2} should be undertaken only when absolutely necessary."
        ],
        'contradiction': [
            "Current medical guidelines do not explicitly restrict the concurrent administration of {e1} and {e2}.",
            "There is a lack of formal contraindications regarding the combination of {e1} and {e2} in standard practice.",
            "Routine clinical protocols generally support the co-administration of {e1} and {e2} without excessive caution.",
            "The prevailing literature does not strongly advise against utilizing {e1} and {e2} within the same regimen.",
            "Intensive clinical monitoring is rarely deemed necessary when combining {e1} and {e2}.",
            "It is generally acceptable to co-prescribe {e1} and {e2} without mandatory empirical dosage adjustments.",
            "Evidence does not support the necessity of seeking alternative agents when considering both {e1} and {e2}.",
            "Formal warnings regarding the simultaneous use of {e1} and {e2} are notably absent from major compendia.",
            "The necessity for high-frequency diagnostic surveillance when pairing {e1} and {e2} remains unproven.",
            "Clinicians are not typically discouraged from initiating therapies that involve both {e1} and {e2}.",
            "Current medical guidelines do not explicitly restrict the concurrent administration of {e1} and {e2}.",
            "There is a lack of formal contraindications regarding the combination of {e1} and {e2} in standard practice.",
            "Routine clinical protocols generally support the co-administration of {e1} and {e2} without excessive caution.",
            "The prevailing literature does not strongly advise against utilizing {e1} and {e2} within the same regimen.",
            "Intensive clinical monitoring is rarely deemed necessary when combining {e1} and {e2}.",
            "It is generally acceptable to co-prescribe {e1} and {e2} without mandatory empirical dosage adjustments.",
            "Evidence does not support the necessity of seeking alternative agents when considering both {e1} and {e2}.",
            "Formal warnings regarding the simultaneous use of {e1} and {e2} are notably absent from major compendia.",
            "The necessity for high-frequency diagnostic surveillance when pairing {e1} and {e2} remains unproven.",
            "Clinicians are not typically discouraged from initiating therapies that involve both {e1} and {e2}."
        ],
        'neutral': [
            "Continuing medical education modules frequently update the prescribing criteria for drugs like {e1} and {e2}.",
            "Pharmaceutical symposiums often dedicate sessions to the broad class properties of {e1} and {e2}.",
            "The raw active ingredients for {e1} and {e2} undergo rigorous quality assurance testing prior to distribution.",
            "National drug shortage databases track the regional supply levels of both {e1} and {e2}.",
            "Independent research institutions regularly publish retrospective analyses featuring {e1} and {e2}.",
            "Manufacturer co-pay assistance programs aim to improve patient access to therapies including {e1} and {e2}.",
            "The foundational primary literature for {e1} and {e2} forms a core part of pharmacy school curricula.",
            "International regulatory bodies maintain dedicated advisory panels to review data on compounds like {e1} and {e2}.",
            "The commercial launch dates for {e1} and {e2} varied significantly across different global markets.",
            "Clinical trial methodologies for {e1} and {e2} adhere strictly to standardized ethical guidelines.",
            "{e1} and {e2} are both referenced in continuing medical education materials.",
            "{e2} appears on several national essential medicines lists along with {e1}.",
            "The dosage strengths available for {e1} differ from those marketed for {e2}.",
            "{e1} and {e2} are included in hospital inventory management systems.",
            "Several clinical trials have independently evaluated {e1} and {e2}.",
            "{e1} is distributed in blister packs, whereas {e2} may be supplied in bottles.",
            "Prescription trends for {e1} and {e2} vary substantially by geographic region.",
            "{e2} is more frequently dispensed in community pharmacies than {e1}.",
            "Both {e1} and {e2} have generic and branded formulations available.",
            "The therapeutic indications for {e1} and {e2} belong to different medical specialties."
        ]
    },
    'int': {
        'entailment': [
            "A broadly recognized drug-drug interaction occurs upon the simultaneous ingestion of {e1} and {e2}.",
            "The medical literature consistently notes a generalized pharmacological interplay between {e1} and {e2}.",
            "Co-administration of {e1} and {e2} invariably leads to an unspecified interactive effect.",
            "Major clinical databases routinely flag an interaction alert for the {e1} and {e2} pairing.",
            "Substantial empirical evidence supports the presence of a generic interaction between {e1} and {e2}.",
            "Combining {e1} and {e2} is formally classified as an interactive therapeutic regimen.",
            "There is a documented biological interplay of clinical note when combining {e1} and {e2}.",
            "The concomitant use of {e1} and {e2} results in a broad systemic interaction.",
            "An interactive relationship between {e1} and {e2} is acknowledged in standard medical compendia.",
            "Clinical pharmacology models predict a generalized interaction following the co-administration of {e1} and {e2}.",
            "An interaction between {e1} and {e2} has been documented.",
            "{e1} is known to interact with {e2} under certain clinical conditions.",
            "Evidence suggests that {e1} and {e2} should be considered interacting agents.",
            "A clinically relevant interaction may occur when {e1} is combined with {e2}.",
            "Reports indicate the presence of an interaction involving {e1} and {e2}.",
            "The concomitant use of {e1} with {e2} has interaction potential.",
            "{e1} and {e2} are recognized as having a drug-drug interaction.",
            "There are established concerns regarding interactions between {e1} and {e2}.",
            "Interaction data exist for the co-administration of {e1} and {e2}.",
            "Combining {e1} and {e2} has been associated with documented interaction findings."
        ],
        'contradiction': [
            "Current evidence does not firmly establish a generic pharmacological interaction between {e1} and {e2}.",
            "There is limited consensus in the literature regarding a definitive biological interplay between {e1} and {e2}.",
            "Routine clinical databases frequently fail to flag an interaction alert for the combination of {e1} and {e2}.",
            "The presumed interactive relationship between {e1} and {e2} lacks robust empirical backing.",
            "It is debatable whether a clinically meaningful interaction truly occurs between {e1} and {e2}.",
            "Broad systemic interactions resulting from the co-administration of {e1} and {e2} are rarely reported.",
            "Extensive systematic reviews provide insufficient support for an unspecified interaction involving {e1} and {e2}.",
            "The pharmacological profiles of {e1} and {e2} do not reliably predict a generalized interactive effect.",
            "Reports of an interactive therapeutic dynamic between {e1} and {e2} are largely considered inconclusive.",
            "Modern compendia offer sparse evidence to validate a generic interaction classification for {e1} and {e2}.",
            "Current evidence does not firmly establish a generic pharmacological interaction between {e1} and {e2}.",
            "There is limited consensus in the literature regarding a definitive biological interplay between {e1} and {e2}.",
            "Routine clinical databases frequently fail to flag an interaction alert for the combination of {e1} and {e2}.",
            "The presumed interactive relationship between {e1} and {e2} lacks robust empirical backing.",
            "It is debatable whether a clinically meaningful interaction truly occurs between {e1} and {e2}.",
            "Broad systemic interactions resulting from the co-administration of {e1} and {e2} are rarely reported.",
            "Extensive systematic reviews provide insufficient support for an unspecified interaction involving {e1} and {e2}.",
            "The pharmacological profiles of {e1} and {e2} do not reliably predict a generalized interactive effect.",
            "Reports of an interactive therapeutic dynamic between {e1} and {e2} are largely considered inconclusive.",
            "Modern compendia offer sparse evidence to validate a generic interaction classification for {e1} and {e2}."
        ],
        'neutral': [
            "Two-dimensional chemical structures for {e1} and {e2} are readily available in open-source chemistry databases.",
            "Both {e1} and {e2} are frequently the subject of large-scale observational cohort studies.",
            "Advanced mass spectrometry techniques are routinely employed to verify the purity of {e1} and {e2}.",
            "The official pharmaceutical monographs detailing {e1} and {e2} undergo periodic revision.",
            "Legal classifications under federal drug laws apply distinct scheduling criteria to {e1} and {e2}.",
            "The procurement costs for the raw materials needed to synthesize {e1} and {e2} fluctuate annually.",
            "Corporate research and development divisions hold the original intellectual property rights for {e1} and {e2}.",
            "{e1} and {e2} are utilized extensively within the scope of human medical practice.",
            "The standardized nomenclature for {e1} and {e2} is governed by international naming councils.",
            "Peer-reviewed journals frequently feature methodological papers discussing the isolation of {e1} and {e2}.",
            "Two-dimensional chemical structures for {e1} and {e2} are readily available in open-source chemistry databases.",
            "Both {e1} and {e2} are frequently the subject of large-scale observational cohort studies.",
            "Advanced mass spectrometry techniques are routinely employed to verify the purity of {e1} and {e2}.",
            "The official pharmaceutical monographs detailing {e1} and {e2} undergo periodic revision.",
            "Legal classifications under federal drug laws apply distinct scheduling criteria to {e1} and {e2}.",
            "The procurement costs for the raw materials needed to synthesize {e1} and {e2} fluctuate annually.",
            "Corporate research and development divisions hold the original intellectual property rights for {e1} and {e2}.",
            "{e1} and {e2} are utilized extensively within the scope of human medical practice.",
            "The standardized nomenclature for {e1} and {e2} is governed by international naming councils.",
            "Peer-reviewed journals frequently feature methodological papers discussing the isolation of {e1} and {e2}."
        ]
    }
}


## 2 · XML parsing

**Fix:** raises `FileNotFoundError` with the offending path instead of silently returning an empty `DataFrame`, so a wrong `DDICorpus` path fails loudly here instead of surfacing as a confusing downstream error.

In [3]:
def parse_ddi_xml_dir(dir_path):
    if not os.path.isdir(dir_path):
        raise FileNotFoundError(
            f"DDI corpus directory not found: '{dir_path}'. "
            f"Check DDICorpus is extracted next to this notebook and the path is correct."
        )

    data = []
    xml_files = [f for f in os.listdir(dir_path) if f.endswith(".xml")]
    if not xml_files:
        raise FileNotFoundError(f"No .xml files found in '{dir_path}'.")

    for filename in xml_files:
        file_path = os.path.join(dir_path, filename)
        tree = ET.parse(file_path)
        root = tree.getroot()

        doc_id = root.attrib.get('id', filename.replace('.xml', ''))

        for sentence in root.findall('sentence'):
            sent_id = sentence.attrib.get('id')
            sent_text = sentence.attrib.get('text')

            entities = {}
            for entity in sentence.findall('entity'):
                e_id = entity.attrib.get('id')
                entities[e_id] = {
                    'text': entity.attrib.get('text'),
                    'type': entity.attrib.get('type')
                }

            for pair in sentence.findall('pair'):
                ddi_bool = pair.attrib.get('ddi')

                data.append({
                    'doc_id': doc_id,
                    'sentence_id': sent_id,
                    'sentence_text': sent_text,
                    'pair_id': pair.attrib.get('id'),
                    'e1_text': entities.get(pair.attrib.get('e1'), {}).get('text', 'UNKNOWN'),
                    'e2_text': entities.get(pair.attrib.get('e2'), {}).get('text', 'UNKNOWN'),
                    'interaction': 1 if ddi_bool == 'true' else 0,
                    'interaction_type': pair.attrib.get('type', 'none')
                })

    return pd.DataFrame(data)


## 3 · Build synthetic NLI pairs

**Fix:** every generated row now carries `original_id` (a stable id derived from the source
`pair_id`, prefixed by scenario type so entailment/contradiction/neutral copies of the same
underlying DDI pair get distinguishable-but-traceable ids), a `scenario` column that mirrors
`label` (for schema parity with the RAG-generated test set, whose `scenario` and `label` are
*not* the same thing — see `1_AugmentResponses.ipynb`), and `entity_valid=True` (trivially true
here — every hypothesis is templated around real corpus drug names).


In [4]:
_id_counter = itertools.count()

def _new_id(pair_id, tag):
    return f"{pair_id}__{tag}__{next(_id_counter)}"


def create_synthetic_nli_pairs(df):
    nli_data = []
    for _, row in df.iterrows():
        premise = row['premise']
        e1 = row['e1_text']
        e2 = row['e2_text']
        label = row['interaction_type']
        pair_id = row['pair_id']

        if label != 'none':
            true_templates = random.sample(augmented_templates[label]['entailment'], k=2)
            false_templates = random.sample(augmented_templates[label]['contradiction'], k=2)
            neutral_templates = random.sample(augmented_templates[label]['neutral'], k=2)

            for t in true_templates:
                nli_data.append({
                    'original_id': _new_id(pair_id, 'ent'),
                    'premise': premise, 'hypothesis': t.format(e1=e1, e2=e2),
                    'label': 'entailment', 'scenario': 'entailment',
                    'e1_text': e1, 'e2_text': e2, 'entity_valid': True,
                })
            for t in false_templates:
                nli_data.append({
                    'original_id': _new_id(pair_id, 'con'),
                    'premise': premise, 'hypothesis': t.format(e1=e1, e2=e2),
                    'label': 'contradiction', 'scenario': 'contradiction',
                    'e1_text': e1, 'e2_text': e2, 'entity_valid': True,
                })
            for t in neutral_templates:
                nli_data.append({
                    'original_id': _new_id(pair_id, 'neu'),
                    'premise': premise, 'hypothesis': t.format(e1=e1, e2=e2),
                    'label': 'neutral', 'scenario': 'neutral',
                    'e1_text': e1, 'e2_text': e2, 'entity_valid': True,
                })
    return pd.DataFrame(nli_data)


## 4 · Assemble train/val split

**Note on fake_drug:** this holistic training set is intentionally 3-way
(`entailment` / `contradiction` / `neutral`) — there is no synthesized `fake_drug` class here.
That's a deliberate design choice, not an oversight: fake-drug/hallucinated-entity detection is a
**deterministic lookup problem** ("is this drug name real?"), not a semantic-inference problem,
so it doesn't belong in the same softmax as entailment/contradiction/neutral. It's evaluated
separately via the `entity_valid` flag, which only becomes non-trivial in the RAG-generated test
set (`1_AugmentResponses.ipynb`), where an LLM is explicitly prompted to substitute a fabricated
drug name. `MAIN_NB`'s previous 4-way `LABEL_MAP = {entailment, contradiction, neutral, fake_drug}`
should be reduced back to the 3-way scheme this proposal's methodology (§2.1.7, §3.4.2) describes,
with `entity_valid` handled as its own pre-filter — that change happens in the `MAIN_NB` pass.

**Other fix:** the previous version's `'none'`-labelled sentences (true non-interacting DDI pairs)
were turned into synthetic *neutral* NLI examples using a randomly chosen **entailment** template
formatted with `e1`/`e2` — i.e. wrapping a genuinely non-interacting pair in language that
asserts an interaction, then calling the result "neutral". That's a label/content mismatch
(the hypothesis text asserts an interaction while being trained as the neutral class). This
version instead samples from the **neutral** template bank for the same reason the labelled
branch above does — the hypothesis text should actually describe neutral/irrelevant content,
not asserted interactions, to match the `neutral` label it's given.


In [5]:
def build_train_val_split(db_path, ml_path):
    df_db = parse_ddi_xml_dir(db_path)
    df_db['source_domain'] = '[DRUGBANK]'
    df_ml = parse_ddi_xml_dir(ml_path)
    df_ml['source_domain'] = '[MEDLINE]'

    df_raw = pd.concat([df_db, df_ml], ignore_index=True)
    df_raw['premise'] = df_raw['source_domain'] + " " + df_raw['sentence_text']

    df_nli = create_synthetic_nli_pairs(df_raw)
    df_none = df_raw[df_raw['interaction_type'] == 'none'].copy()
    df_none_unique = df_none.drop_duplicates(subset=['premise'])

    sample_size = min(len(df_nli) // 3, len(df_none_unique))
    df_none_sampled = df_none_unique.sample(n=sample_size, random_state=42)

    # FIX: sample from the NEUTRAL template bank (previously sampled from the
    # entailment bank, which asserts an interaction — a content/label mismatch
    # for rows that are meant to represent neutral/non-interacting content).
    all_neutral_templates = (
        augmented_templates['effect']['neutral'] +
        augmented_templates['mechanism']['neutral'] +
        augmented_templates['advise']['neutral'] +
        augmented_templates['int']['neutral']
    )

    none_data = []
    for _, row in df_none_sampled.iterrows():
        n_template = random.choice(all_neutral_templates)
        none_data.append({
            'original_id': _new_id(row['pair_id'], 'none_neu'),
            'e1_text': row['e1_text'],
            'e2_text': row['e2_text'],
            'premise': row['premise'],
            'hypothesis': n_template.format(e1=row['e1_text'], e2=row['e2_text']),
            'label': 'neutral',
            'scenario': 'neutral',
            'entity_valid': True,
        })

    final_df = pd.concat([df_nli, pd.DataFrame(none_data)], ignore_index=True)

    final_df = final_df[[
        'original_id', 'e1_text', 'e2_text', 'premise', 'hypothesis',
        'label', 'scenario', 'entity_valid',
    ]]

    return final_df.sample(frac=1, random_state=42).reset_index(drop=True)


## 5 · Raw test manifest

Unchanged in logic. This is consumed directly by `1_AugmentResponses.ipynb` (fixed to actually read this file instead of re-parsing the XML a second time with slightly different filtering logic — see that notebook's changelog).

In [6]:
def build_raw_test_sources(db_path, ml_path):
    df_db = parse_ddi_xml_dir(db_path)
    df_db['source_domain'] = '[DRUGBANK]'
    df_ml = parse_ddi_xml_dir(ml_path)
    df_ml['source_domain'] = '[MEDLINE]'

    df_raw = pd.concat([df_db, df_ml], ignore_index=True)
    df_raw['premise'] = df_raw['source_domain'] + " " + df_raw['sentence_text']
    df_raw['original_id'] = df_raw['pair_id']

    test_manifest = df_raw[[
        'original_id', 'e1_text', 'e2_text', 'premise', 'interaction_type', 'source_domain',
    ]].copy()
    return test_manifest


In [9]:
import random
from sklearn.model_selection import train_test_split

train_db = "../DDICorpus/Train/DrugBank"
train_ml = "../DDICorpus/Train/MedLine"
test_db  = "../DDICorpus/Test/Test for DDI Extraction task/DrugBank"
test_ml  = "../DDICorpus/Test/Test for DDI Extraction task/MedLine"

full_train_df    = build_train_val_split(train_db, train_ml)
test_manifest_df = build_raw_test_sources(test_db, test_ml)

train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.15,
    stratify=full_train_df['label'],
    random_state=42
)

print(f"Holistic train: {len(train_df):,}  val: {len(val_df):,}")
print("\nTrain scenario distribution:")
print(train_df['scenario'].value_counts())
print(f"\nRaw test manifest (for RAG augmentation): {len(test_manifest_df):,} DDI pairs")


Holistic train: 23,028  val: 4,064

Train scenario distribution:
scenario
neutral          9360
entailment       6834
contradiction    6834
Name: count, dtype: int64

Raw test manifest (for RAG augmentation): 5,716 DDI pairs


## 6 · Save outputs

**Fix:** filenames renamed to `holistic_train.csv` / `holistic_val.csv` to match the `{prefix}_train.csv`/`{prefix}_val.csv` convention `MAIN_NB.NLIDataBundle` reads (`prefix="holistic"`). The old names (`train_augmented_nli.csv`, `val_augmented_nli.csv`) required an undocumented manual rename before `MAIN_NB` could use them.

In [10]:
os.makedirs("data", exist_ok=True)
train_df.to_csv("data/holistic_train.csv", index=False)
val_df.to_csv("data/holistic_val.csv", index=False)
test_manifest_df.to_csv("test_raw_manifest.csv", index=False)

print("Saved:")
print("  data/holistic_train.csv   ", len(train_df))
print("  data/holistic_val.csv     ", len(val_df))
print("  test_raw_manifest.csv     ", len(test_manifest_df), " (feeds 1_AugmentResponses.ipynb)")


Saved:
  data/holistic_train.csv    23028
  data/holistic_val.csv      4064
  test_raw_manifest.csv      5716  (feeds 1_AugmentResponses.ipynb)


## Changelog

| # | Issue in original notebook | Fix |
|---|---|---|
| 1 | No `original_id` column | Added stable, traceable `original_id` per generated row |
| 2 | No `scenario` column, but `MAIN_NB.NLIDataBundle.summary()` reads `df['scenario']` | Added `scenario` (== `label` for this holistic set) |
| 3 | `'none'` (non-interacting) DDI pairs were wrapped in **entailment**-bank templates but labelled `neutral` — hypothesis text contradicts its own label | Now sampled from the **neutral** template bank |
| 4 | `parse_ddi_xml_dir` silently returns an empty `DataFrame` on a bad path → confusing downstream `KeyError` | Raises `FileNotFoundError` immediately with the bad path |
| 5 | Output filenames (`train_augmented_nli.csv`) didn't match what `MAIN_NB` expects (`holistic_train.csv`) | Renamed outputs to match `MAIN_NB`'s `{prefix}_train.csv` convention |
| 6 | No `entity_valid` column | Added (`True` here by construction) for schema parity with `holistic_test.csv` |
| 7 | Fake-drug detection ambiguity: original run's `holistic_train.csv` apparently *did* contain a 4th `fake_drug` scenario from an undocumented step not present in any of these three notebooks | Explicitly scoped out — holistic training stays 3-way; fake-drug/hallucination detection is handled by `entity_valid`, evaluated only at test time |
